# PatchCore: fit, save/load, and raw validation

Use the existing project environment. This notebook calls repository scripts. It does not install packages or download weights. The fit/raw cells do not evaluate test data; the final cells only read an already saved evaluation. See `docs/patchcore-guide.md`.


In [ ]:
from pathlib import Path
import subprocess
import sys
import json
from datetime import datetime
from zoneinfo import ZoneInfo

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook inside the repository")
    ROOT = ROOT.parent

CATEGORY = "tile"  # metal_nut requires the T05 config
SMOKE = True  # set False for a complete good-train fit
EXISTING_RUN_ID = None  # set to inspect an existing artifact without fitting again
FOLD = {"tile": "holdout-crack", "metal_nut": "holdout-scratch"}[CATEGORY]
CONFIG = ROOT / f"configs/anomaly/patchcore_{CATEGORY}.yaml"
MANIFEST = ROOT / f"splits/seed42/custom_comparison_{CATEGORY}_{FOLD.removeprefix('holdout-')}.csv"
RUN_ID = EXISTING_RUN_ID or f"{datetime.now(ZoneInfo('Asia/Bangkok')):%Y%m%d-%H%M%S}-seed42-{'smoke-' if SMOKE else ''}{FOLD}"
ARTIFACT = ROOT / "artifacts/custom_comparison/patchcore" / CATEGORY / RUN_ID
OUTPUT = ROOT / "outputs/custom_comparison/patchcore" / CATEGORY / RUN_ID
print(CONFIG, RUN_ID, sep="\n")


## Fit good images only

The approved manifest and the explicitly downloaded pretrained ResNet-18 checkpoint must already be present. SMOKE uses four good images. Full tile uses 184 good images; full metal_nut uses 176.


In [ ]:
if EXISTING_RUN_ID is None:
    command = [sys.executable, "scripts/train_anomaly.py", "--config", str(CONFIG), "--run-id", RUN_ID]
    if SMOKE:
        command.append("--smoke-test")
    subprocess.run(command, cwd=ROOT, check=True)
metadata = json.loads((ARTIFACT / "metadata.json").read_text(encoding="utf-8"))
print(json.dumps({k: metadata[k] for k in ["run_id", "training_seconds", "memory_bank_shape", "training_manifest", "status"]}, indent=2))


## Reload in another process and export validation maps

No PASS/DEFECT labels or official metrics are produced before shared calibration is integrated. The command rejects an existing output directory.


In [ ]:
RAW_OUTPUT = OUTPUT / "raw-validation"
if not RAW_OUTPUT.exists():
    subprocess.run([sys.executable, "scripts/predict.py", "--artifact-dir", str(ARTIFACT), "--manifest", str(MANIFEST), "--dataset-root", str(ROOT / "datasets"), "--device", "cuda", "--output-dir", str(RAW_OUTPUT)], cwd=ROOT, check=True)
print((RAW_OUTPUT / "summary.json").read_text())


In [ ]:
from IPython.display import display
from PIL import Image

for path in sorted((RAW_OUTPUT / "visualizations").glob("*.png")):
    print(path.name)
    with Image.open(path) as image:
        display(image.copy())


## Next: shared H03 integration

Register `PatchcorePredictor` with the reviewed shared loader, calibrate on normal validation, then lock config/checkpoint/thresholds before evaluating test. Follow the H02/H03 and U-Net artifact handoff checklist in `docs/patchcore-guide.md`.


## Read an existing shared evaluation
This cell does not run test inference or change thresholds. See `docs/results/shared-evaluation-day21.md` for the locked baseline and U-Net comparison.


In [ ]:
EVAL_DIR = OUTPUT / f"eval-{MANIFEST.stem}"
if (EVAL_DIR / "metrics.json").is_file():
    metrics = json.loads((EVAL_DIR / "metrics.json").read_text())
    print(json.dumps({k: metrics[k] for k in ["image", "pixel", "groups"]}, indent=2))
else:
    print("No saved evaluation for this run. Calibrate and lock the experiment before test evaluation.")
